In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import pipeline
import re
import json
import torch

In [5]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [10]:
model.device

device(type='cuda', index=0)

### Inference With Tools

In [8]:
tools = [
    {
        "name": "get_weather",
        "description": "Get weather for a city",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string"
                }
            },
            "required": ["city"]
        }
    }
]

messages = [
    {"role": "user", "content": "what is the weather in Jakarta?"}
]

In [9]:
text = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True
)

In [11]:
inputs = tokenizer(text, return_tensors="pt").to(model.device)

In [14]:
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True
    )

In [16]:
inputs['input_ids']

tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
            382,      2,  13852,    271,   2610,   1231,   1618,    825,    476,
            803,   5746,    311,   7789,    448,    279,   1196,   3239,    382,
           2610,    525,   3897,    448,    729,  32628,   2878,    366,  15918,
           1472,  15918,     29,  11874,   9492,    510,     27,  15918,    397,
           4913,    606,    788,    330,    455,  69364,    497,    330,   4684,
            788,    330,   1949,   9104,    369,    264,   3283,    497,    330,
          13786,    788,   5212,   1313,    788,    330,   1700,    497,    330,
          13193,    788,   5212,   8926,    788,   5212,   1313,    788,    330,
            917,   9207,   2137,    330,   6279,    788,   4383,   8926,   1341,
          11248,    522,  15918,   1339,   2461,   1817,    729,   1618,     11,
            470,    264,   2

In [15]:
outputs

tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
            382,      2,  13852,    271,   2610,   1231,   1618,    825,    476,
            803,   5746,    311,   7789,    448,    279,   1196,   3239,    382,
           2610,    525,   3897,    448,    729,  32628,   2878,    366,  15918,
           1472,  15918,     29,  11874,   9492,    510,     27,  15918,    397,
           4913,    606,    788,    330,    455,  69364,    497,    330,   4684,
            788,    330,   1949,   9104,    369,    264,   3283,    497,    330,
          13786,    788,   5212,   1313,    788,    330,   1700,    497,    330,
          13193,    788,   5212,   8926,    788,   5212,   1313,    788,    330,
            917,   9207,   2137,    330,   6279,    788,   4383,   8926,   1341,
          11248,    522,  15918,   1339,   2461,   1817,    729,   1618,     11,
            470,    264,   2

In [18]:
len(inputs['input_ids'][0])

159

In [19]:
len(outputs[0])

180

In [21]:
output_tokens_count = len(outputs[0]) - len(inputs['input_ids'][0])

In [26]:
input_tokens_count = len(inputs['input_ids'][0])
output_tokens = outputs[0][input_tokens_count:]

In [27]:
output_tokens

tensor([151657,    198,   4913,    606,    788,    330,    455,  69364,    497,
           330,  16370,    788,   5212,   8926,    788,    330,  89272,  24969,
         95642, 151658, 151645], device='cuda:0')

In [28]:
result = tokenizer.decode(output_tokens, skip_special_tokens=False)

In [29]:
result

'<tool_call>\n{"name": "get_weather", "arguments": {"city": "Jakarta"}}\n</tool_call><|im_end|>'

In [30]:
import re
import json
from typing import Optional, Dict, Any

def parse_tool_call(text: str) -> Optional[Dict[str, Any]]:
    pattern = r"<tool_call>\s*(\{.*?\})\s*</tool_call><|im_end|>"
    match = re.search(pattern, text, re.DOTALL)
    if not match:
        return None

    try:
        return json.loads(match.group(1))
    except json.JSONDecodeError:
        return None

In [31]:
tool_call = parse_tool_call(result)

In [32]:
def get_weather(city):
    return f"The weather in {city} is 30°C and sunny."

tool_map = {
    "get_weather": get_weather
}

In [33]:
if tool_call:
    tool_name = tool_call["name"]
    tool_args = tool_call["arguments"]

    func = tool_map.get(tool_name)

    if func is None:
        raise ValueError(f"Tool '{tool_name}' not found")

    tool_result = func(**tool_args)

In [34]:
tool_result

'The weather in Jakarta is 30°C and sunny.'

In [35]:
messages

[{'role': 'user', 'content': 'what is the weather in Jakarta?'}]

In [37]:
new_messages = [
    {"role": "user", "content": "what is the weather in Jakarta?"},
    {
        "role": "assistant",
        "content": "",
        "tool_calls": [
            tool_call
        ]
    },
    {
        "role": "tool",
        "content": tool_result
    }
]

In [38]:
new_messages

[{'role': 'user', 'content': 'what is the weather in Jakarta?'},
 {'role': 'assistant',
  'content': '',
  'tool_calls': [{'name': 'get_weather', 'arguments': {'city': 'Jakarta'}}]},
 {'role': 'tool', 'content': 'The weather in Jakarta is 30°C and sunny.'}]

In [42]:
text = tokenizer.apply_chat_template(
    new_messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True
)

In [44]:
inputs = tokenizer(text, return_tensors="pt").to(model.device)

In [45]:
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True
)

In [46]:
result = tokenizer.decode(outputs[0], skip_special_tokens=False)

In [48]:
print(result)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"name": "get_weather", "description": "Get weather for a city", "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
what is the weather in Jakarta?<|im_end|>
<|im_start|>assistant
<tool_call>
{"name": "get_weather", "arguments": {"city": "Jakarta"}}
</tool_call><|im_end|>
<|im_start|>user
<tool_response>
The weather in Jakarta is 30°C and sunny.
</tool_response><|im_end|>
<|im_start|>assistant
The current weather in Jakarta is 30°C and it's sunny.<|im_end|>


In [49]:
len(inputs['input_ids'][0])

209

In [50]:
len(outputs[0])

225

In [51]:
input_tokens_count = len(inputs['input_ids'][0])
output_tokens = outputs[0][input_tokens_count:]

In [52]:
result = tokenizer.decode(output_tokens, skip_special_tokens=False)

In [53]:
result

"The current weather in Jakarta is 30°C and it's sunny.<|im_end|>"